<a href="https://colab.research.google.com/github/1p2a3r4a/flyrank-project/blob/main/Copy_of_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1p2a3r4a/flyrank-project/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"{rel}/fact_content_daily_performance/**/*.parquet"

con.sql(f"SELECT COUNT(*) FROM read_parquet('{TABLE}')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)

schema_df = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{TABLE}')").df()
print(schema_df[['column_name', 'column_type']].to_string())

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [ ]:
con.sql(f"SELECT * FROM read_parquet('{TABLE}') LIMIT 5").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [ ]:
# CLAIM: one row = one (content_hash_id, client_hash_id, report_date)
con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS c
    FROM read_parquet('{TABLE}')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬─────────────────────────┬─────────────┬───────┐
│     content_hash_id      │     client_hash_id      │ report_date │   c   │
│         varchar          │         varchar         │    date     │ int64 │
├──────────────────────────┼─────────────────────────┼─────────────┼───────┤
│ content_b7d3cee2b4f1e8f4 │ client_a22068e339bf95f5 │ 2026-06-13  │     2 │
│ content_41187afd3aee49b5 │ client_1a8bf67cad4ee525 │ 2026-06-13  │     2 │
│ content_3da2cf6ca590051b │ client_1a8bf67cad4ee525 │ 2026-06-13  │     2 │
│ content_5c5b96cd4d1829d8 │ client_1a730cb2640a1abf │ 2026-06-13  │     2 │
│ content_10da8298298f2796 │ client_8ddc46da5414ffd8 │ 2026-06-14  │     2 │
└──────────────────────────┴─────────────────────────┴─────────────┴───────┘

In [ ]:
con.sql(f"""
    SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest,
           COUNT(DISTINCT report_date) AS n_distinct_dates,
           COUNT(DISTINCT month) AS n_distinct_months
    FROM read_parquet('{TABLE}')
""")

┌────────────┬────────────┬──────────────────┬───────────────────┐
│  earliest  │   latest   │ n_distinct_dates │ n_distinct_months │
│    date    │    date    │      int64       │       int64       │
├────────────┼────────────┼──────────────────┼───────────────────┤
│ 2025-01-27 │ 2026-06-30 │              520 │                18 │
└────────────┴────────────┴──────────────────┴───────────────────┘

In [ ]:
# Is report_date really daily, or does the table only carry one row per month?
con.sql(f"""
    SELECT month, MIN(report_date) AS first_date, MAX(report_date) AS last_date,
           COUNT(DISTINCT report_date) AS n_dates_in_month
    FROM read_parquet('{TABLE}')
    GROUP BY month
    ORDER BY month
    LIMIT 15
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┬──────────────────┐
│  month  │ first_date │ last_date  │ n_dates_in_month │
│ varchar │    date    │    date    │      int64       │
├─────────┼────────────┼────────────┼──────────────────┤
│ 2025-01 │ 2025-01-27 │ 2025-01-31 │                5 │
│ 2025-02 │ 2025-02-01 │ 2025-02-28 │               28 │
│ 2025-03 │ 2025-03-01 │ 2025-03-31 │               31 │
│ 2025-04 │ 2025-04-01 │ 2025-04-30 │               30 │
│ 2025-05 │ 2025-05-01 │ 2025-05-31 │               31 │
│ 2025-06 │ 2025-06-01 │ 2025-06-30 │               30 │
│ 2025-07 │ 2025-07-01 │ 2025-07-31 │               31 │
│ 2025-08 │ 2025-08-01 │ 2025-08-31 │               31 │
│ 2025-09 │ 2025-09-01 │ 2025-09-30 │               30 │
│ 2025-10 │ 2025-10-01 │ 2025-10-31 │               31 │
│ 2025-11 │ 2025-11-01 │ 2025-11-30 │               30 │
│ 2025-12 │ 2025-12-01 │ 2025-12-31 │               31 │
│ 2026-01 │ 2026-01-01 │ 2026-01-31 │               31 │
│ 2026-02 │ 2026-02-01 │ 2026-0

In [ ]:
con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           MIN(report_date) AS first_seen, MAX(report_date) AS last_seen,
           COUNT(*) AS n_rows
    FROM read_parquet('{TABLE}')
    GROUP BY content_hash_id, client_hash_id
    ORDER BY first_seen
    LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬─────────────────────────┬────────────┬────────────┬────────┐
│     content_hash_id      │     client_hash_id      │ first_seen │ last_seen  │ n_rows │
│         varchar          │         varchar         │    date    │    date    │ int64  │
├──────────────────────────┼─────────────────────────┼────────────┼────────────┼────────┤
│ content_3b70a18ea133b2bb │ client_9958f0a7ae1df715 │ 2025-01-27 │ 2026-06-30 │    520 │
│ content_fe8e8155ce1d47a2 │ client_9958f0a7ae1df715 │ 2025-01-27 │ 2026-06-30 │    476 │
│ content_b4462a1b90640058 │ client_9958f0a7ae1df715 │ 2025-01-27 │ 2026-06-30 │    507 │
│ content_c899aef92518c714 │ client_9958f0a7ae1df715 │ 2025-01-27 │ 2026-06-30 │    520 │
│ content_c782fa8abd4fce5e │ client_9958f0a7ae1df715 │ 2025-01-27 │ 2026-06-30 │    520 │
│ content_658f53fa439c66ca │ client_9958f0a7ae1df715 │ 2025-01-27 │ 2026-06-30 │    429 │
│ content_5ca1b43f9a4d0b01 │ client_9958f0a7ae1df715 │ 2025-01-27 │ 2026-06-30 │    520 │
│ content_

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
con.sql(f"""
    SELECT client_has_gsc, gsc_data_available, COUNT(*) AS n,
           AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS pct_null_gsc_clicks
    FROM read_parquet('{TABLE}')
    GROUP BY client_has_gsc, gsc_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬────────────────────┬──────────┬─────────────────────┐
│ client_has_gsc │ gsc_data_available │    n     │ pct_null_gsc_clicks │
│    boolean     │      boolean       │  int64   │       double        │
├────────────────┼────────────────────┼──────────┼─────────────────────┤
│ true           │ false              │ 49767598 │                 0.0 │
│ false          │ NULL               │    98006 │                 1.0 │
│ true           │ true               │ 28970051 │                 0.0 │
└────────────────┴────────────────────┴──────────┴─────────────────────┘

In [ ]:
con.sql(f"""
    SELECT client_has_ga4, ga4_data_available, COUNT(*) AS n
    FROM read_parquet('{TABLE}')
    GROUP BY client_has_ga4, ga4_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬────────────────────┬──────────┐
│ client_has_ga4 │ ga4_data_available │    n     │
│    boolean     │      boolean       │  int64   │
├────────────────┼────────────────────┼──────────┤
│ true           │ false              │ 44053460 │
│ false          │ false              │  2330413 │
│ false          │ NULL               │ 29635327 │
│ true           │ true               │  2816455 │
└────────────────┴────────────────────┴──────────┘

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
con.sql(f"""
    SELECT
        AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS pct_null_gsc_impressions,
        AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS pct_null_gsc_clicks,
        AVG(CASE WHEN gsc_sum_position IS NULL THEN 1.0 ELSE 0 END) AS pct_null_gsc_position,
        AVG(CASE WHEN scroll_events IS NULL THEN 1.0 ELSE 0 END) AS pct_null_scroll_events
    FROM read_parquet('{TABLE}')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬───────────────────────┬───────────────────────┬────────────────────────┐
│ pct_null_gsc_impressions │  pct_null_gsc_clicks  │ pct_null_gsc_position │ pct_null_scroll_events │
│          double          │        double         │        double         │         double         │
├──────────────────────────┼───────────────────────┼───────────────────────┼────────────────────────┤
│    0.0012431684622903178 │ 0.0012431684622903178 │ 0.0012432699392172235 │     0.3759127389757845 │
└──────────────────────────┴───────────────────────┴───────────────────────┴────────────────────────┘

In [ ]:
con.sql(f"""
    SELECT gsc_data_available, COUNT(*) AS n,
           AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS pct_null_clicks,
           AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS pct_null_impressions
    FROM read_parquet('{TABLE}')
    GROUP BY gsc_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬──────────┬─────────────────┬──────────────────────┐
│ gsc_data_available │    n     │ pct_null_clicks │ pct_null_impressions │
│      boolean       │  int64   │     double      │        double        │
├────────────────────┼──────────┼─────────────────┼──────────────────────┤
│ false              │ 49767598 │             0.0 │                  0.0 │
│ NULL               │    98006 │             1.0 │                  1.0 │
│ true               │ 28970051 │             0.0 │                  0.0 │
└────────────────────┴──────────┴─────────────────┴──────────────────────┘

In [ ]:
con.sql(f"""
    SELECT content_hash_id, client_hash_id, COUNT(*) AS n_rows,
           DATEDIFF('day', MIN(report_date), MAX(report_date)) + 1 AS days_in_window
    FROM read_parquet('{TABLE}')
    GROUP BY content_hash_id, client_hash_id
    ORDER BY n_rows DESC
    LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬─────────────────────────┬────────┬────────────────┐
│     content_hash_id      │     client_hash_id      │ n_rows │ days_in_window │
│         varchar          │         varchar         │ int64  │     int64      │
├──────────────────────────┼─────────────────────────┼────────┼────────────────┤
│ content_a64143f6e4a21ffe │ client_9958f0a7ae1df715 │    520 │            520 │
│ content_4b90d8f71a8d9c59 │ client_9958f0a7ae1df715 │    520 │            520 │
│ content_167aff5fed6593c7 │ client_9958f0a7ae1df715 │    520 │            520 │
│ content_ae5e5fd6edff550f │ client_9958f0a7ae1df715 │    520 │            520 │
│ content_d237e43c93144a57 │ client_9958f0a7ae1df715 │    520 │            520 │
│ content_f73516da99611861 │ client_9958f0a7ae1df715 │    520 │            520 │
│ content_6c7a4022b0992856 │ client_9958f0a7ae1df715 │    520 │            520 │
│ content_c7c1d2e68d9d0964 │ client_9958f0a7ae1df715 │    520 │            520 │
│ content_5a5d7320a8901ea8 │

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
con.sql(f"""
    SELECT client_hash_id,
           BOOL_AND(client_has_gsc) AS always_has_gsc,
           BOOL_AND(client_has_ga4) AS always_has_ga4,
           MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM read_parquet('{TABLE}')
    GROUP BY client_hash_id
    LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬────────────────┬────────────────┬────────────┬────────────┐
│     client_hash_id      │ always_has_gsc │ always_has_ga4 │  earliest  │   latest   │
│         varchar         │    boolean     │    boolean     │    date    │    date    │
├─────────────────────────┼────────────────┼────────────────┼────────────┼────────────┤
│ client_9958f0a7ae1df715 │ true           │ true           │ 2025-01-27 │ 2026-06-30 │
│ client_73cda7b4e4f265ea │ true           │ false          │ 2025-02-11 │ 2026-06-30 │
│ client_fef1a8f436438636 │ true           │ false          │ 2025-03-11 │ 2026-06-30 │
│ client_c182d11e4862a37d │ true           │ true           │ 2025-06-21 │ 2026-06-30 │
│ client_a2eeb8899886adde │ true           │ false          │ 2025-07-06 │ 2026-06-30 │
│ client_62f4a7e64f5e0096 │ true           │ false          │ 2025-06-07 │ 2026-06-30 │
│ client_d211cb07b9059bab │ true           │ false          │ 2025-07-07 │ 2026-06-30 │
│ client_8ae2bfb5aa1ffa1e │ true

In [ ]:
con.sql(f"""
    SELECT client_hash_id, gsc_data_available,
           MIN(report_date) AS first_date_at_this_state
    FROM read_parquet('{TABLE}')
    GROUP BY client_hash_id, gsc_data_available
    ORDER BY client_hash_id, first_date_at_this_state
    LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬────────────────────┬──────────────────────────┐
│     client_hash_id      │ gsc_data_available │ first_date_at_this_state │
│         varchar         │      boolean       │           date           │
├─────────────────────────┼────────────────────┼──────────────────────────┤
│ client_04660893ae39614a │ false              │ 2026-05-22               │
│ client_06d356715a8ff3b6 │ false              │ 2026-04-02               │
│ client_06d356715a8ff3b6 │ true               │ 2026-04-10               │
│ client_0797ff3a1fc9a6a5 │ false              │ 2025-11-05               │
│ client_0797ff3a1fc9a6a5 │ true               │ 2025-11-05               │
│ client_08a6a72ff48e62c0 │ true               │ 2025-09-24               │
│ client_08a6a72ff48e62c0 │ false              │ 2025-11-05               │
│ client_08d2847f24cf89c1 │ true               │ 2025-07-21               │
│ client_08d2847f24cf89c1 │ false              │ 2025-11-05               │
│ client_0b2

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.